# 06 - 完整实验：端到端 MTP 评测流程

## 学习目标
- 编写完整的 experiment.json 配置
- 编写 deploy shell script
- 使用 `framework/runner.py` 运行端到端实验
- 解析和可视化评测结果
- 执行多模型对比评测
- 掌握常见问题排查方法

## 1. 编写 Deploy Script

deploy script 是一个 shell 脚本，负责启动 SGLang 推理服务。

In [ ]:
import os
import json

DATA_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "data")
os.makedirs(DATA_DIR, exist_ok=True)

# ============================================================
# 编写 deploy script
# ============================================================

# 请根据你的环境修改以下变量
MODEL_PATH = "/your/model/path"  # MTP 模型路径
TP_SIZE = 8
DP_SIZE = 1
PORT = 30000

deploy_script = f"""#!/bin/bash
# Deploy script for MTP evaluation
# Model: {os.path.basename(MODEL_PATH)}

python3 -m sglang.launch_server \\
    --model-path {MODEL_PATH} \\
    --tp-size {TP_SIZE} \\
    --dp-size {DP_SIZE} \\
    --enable-dp-attention \\
    --trust-remote-code \\
    --mem-fraction-static 0.75 \\
    --max-running-requests 256 \\
    --cuda-graph-max-bs 32 \\
    --chunked-prefill-size 16384 \\
    --speculative-algorithm EAGLE \\
    --speculative-num-steps=3 \\
    --speculative-eagle-topk=1 \\
    --speculative-num-draft-tokens=4 \\
    --host 0.0.0.0 \\
    --port {PORT}
"""

deploy_script_path = os.path.join(DATA_DIR, "start_model_eagle.sh")
with open(deploy_script_path, "w") as f:
    f.write(deploy_script)
os.chmod(deploy_script_path, 0o755)

print(f"Deploy script saved to: {deploy_script_path}")
print("\nContent:")
print(deploy_script)

## 2. 编写 experiment.json

experiment.json 是 `framework/runner.py` 的输入配置，定义了整个实验的服务和任务。

In [ ]:
# ============================================================
# 编写 experiment.json
# ============================================================

# 实际的 bench 脚本和数据文件路径
# 请根据你的 qf_mtp_eval 安装路径修改
QF_MTP_EVAL_ROOT = "/mnt/cfs_bj_mt/workspace/limengjie03/tool_chain/qf_mtp_eval"

experiment_config = {
    "experiment": {
        "name": "my-first-mtp-eval",
        "continue_on_error": True,  # 某个 task 失败不影响后续
    },
    "service": {
        "command_file": deploy_script_path,
        "ready_port": PORT,
        "ready_url": f"http://127.0.0.1:{PORT}/health",
        "startup_timeout_sec": 300,  # 大模型加载可能需要 5 分钟
    },
    "tasks": [
        {
            "id": "sample_bench_basic",
            "bench": "aiak_bench",
            "args": {
                "port": PORT,
                "num-questions": 100,
                "max-gen-length": 1024,
                "thinking": False,
                "mt": "all",
            },
        },
        {
            "id": "sample_bench_thinking",
            "bench": "math_500",
            "args": {
                "port": PORT,
                "num-questions": 50,
                "max-gen-length": 4096,
                "thinking": True,
            },
        },
    ],
}

experiment_path = os.path.join(DATA_DIR, "experiment.json")
with open(experiment_path, "w") as f:
    json.dump(experiment_config, f, indent=2)

print(f"Experiment config saved to: {experiment_path}")
print("\nConfig:")
print(json.dumps(experiment_config, indent=2))

## 3. 使用 framework/runner.py 运行实验

Runner 是 qf_mtp_eval 的统一执行入口，支持两种子命令：

### 子命令

| 命令 | 用途 |
|------|------|
| `run-bench` | 仅运行 benchmark（假设服务已启动） |
| `run-experiment` | 完整流程：启动服务 → 运行所有 task → 停止服务 |

### 使用方式

```bash
# 完整实验（启动服务 + 评测 + 清理）
python3 evaluation_dev/framework/runner.py run-experiment \
    --config experiment.json \
    --output-dir ./runs/my-experiment

# 仅运行 benchmark（服务已在外部启动）
python3 evaluation_dev/framework/runner.py run-bench \
    --config experiment.json \
    --task-id sample_bench_basic \
    --output-dir ./runs/my-bench
```

In [ ]:
import subprocess

# ============================================================
# 方式 A: 使用 runner.py 运行完整实验
# ============================================================

RUNNER_PATH = os.path.join(QF_MTP_EVAL_ROOT, "evaluation_dev/framework/runner.py")
OUTPUT_DIR = os.path.join(DATA_DIR, "runs", "my-first-experiment")

# 构建命令
runner_cmd = [
    "python3", RUNNER_PATH, "run-experiment",
    "--config", experiment_path,
    "--output-dir", OUTPUT_DIR,
]

print("=== Runner Command ===")
print(" \\\n    ".join(runner_cmd))
print(f"\nOutput directory: {OUTPUT_DIR}")
print("\n(Uncomment below to actually run)")

In [ ]:
# 实际执行（取消注释运行）
# 注意: 这会启动 SGLang 服务并运行评测，需要 GPU 环境

# os.makedirs(OUTPUT_DIR, exist_ok=True)
# 
# process = subprocess.Popen(
#     runner_cmd,
#     stdout=subprocess.PIPE,
#     stderr=subprocess.STDOUT,
#     text=True,
# )
# 
# # Stream output
# for line in process.stdout:
#     print(line, end="")
# 
# process.wait()
# print(f"\nRunner exited with code: {process.returncode}")

## 4. 方式 B: 手动分步执行

如果你想更灵活地控制流程，可以手动分步执行：

1. 在终端启动服务
2. 在 notebook 中运行 benchmark
3. 手动停止服务

In [ ]:
import time
import requests

# ============================================================
# Step 1: 确认服务已启动
# ============================================================
# 在终端执行: bash data/start_model_eagle.sh

def check_server(host="127.0.0.1", port=30000):
    """Check if SGLang server is running."""
    try:
        resp = requests.get(f"http://{host}:{port}/health", timeout=3)
        if resp.status_code == 200:
            print(f"Server is running at {host}:{port}")
            # Get model info
            model_resp = requests.get(f"http://{host}:{port}/v1/models")
            models = model_resp.json()
            print(f"Model: {models.get('data', [{}])[0].get('id', 'unknown')}")
            return True
    except requests.ConnectionError:
        print(f"Server not reachable at {host}:{port}")
    return False

check_server("127.0.0.1", PORT)

In [ ]:
# ============================================================
# Step 2: 运行 Benchmark 脚本
# ============================================================

BENCH_SCRIPT = os.path.join(QF_MTP_EVAL_ROOT, "evaluation_dev/aiak_bench/bench_sglang_eagle.py")
QUESTION_FILE = os.path.join(QF_MTP_EVAL_ROOT, "evaluation_dev/aiak_bench/split/eval_10w_2k.jsonl")
ANSWER_FILE = os.path.join(DATA_DIR, "answers_manual.jsonl")
RESULT_FILE = os.path.join(DATA_DIR, "result_manual.json")

bench_cmd = [
    "python3", BENCH_SCRIPT,
    "--question-file", QUESTION_FILE,
    "--answer-file", ANSWER_FILE,
    "--result-file", RESULT_FILE,
    "--num-questions", "100",
    "--temperature", "0",
    "--max-gen-length", "1024",
    "--parallel", "16",
    "--host", f"http://127.0.0.1",
    "--port", str(PORT),
]

print("Benchmark command:")
print(" \\\n    ".join(bench_cmd))

# Uncomment to run:
# process = subprocess.run(bench_cmd, capture_output=True, text=True)
# print(process.stdout)
# if process.returncode != 0:
#     print(f"ERROR: {process.stderr}")

## 5. 解析评测结果

In [ ]:
# ============================================================
# 解析 result.json
# ============================================================

def load_and_display_result(result_file):
    """Load and pretty-print a result.json file."""
    if not os.path.exists(result_file):
        print(f"Result file not found: {result_file}")
        print("(Run the benchmark first)")
        return None
    
    with open(result_file) as f:
        result = json.load(f)
    
    print("=" * 60)
    print("EVALUATION RESULT")
    print("=" * 60)
    print(f"Task:             {result.get('task', 'N/A')}")
    print(f"Backend:          {result.get('backend', 'N/A')}")
    print(f"Num requests:     {result.get('num_requests', 'N/A')}")
    print(f"Single-turn:      {result.get('single_turn_count', 'N/A')}")
    print(f"Multi-turn:       {result.get('multi_turn_count', 'N/A')}")
    print(f"\n--- Performance ---")
    print(f"Throughput:       {result.get('throughput', 0):.2f} tokens/s")
    print(f"Latency:          {result.get('latency', 0):.2f} s")
    print(f"Accept Length:    {result.get('accept_length', 0):.3f}")
    print("=" * 60)
    
    return result


# Try loading result
load_and_display_result(RESULT_FILE)

In [ ]:
# ============================================================
# 解析 answers.jsonl — 逐样本分析
# ============================================================

def analyze_answers(answer_file, top_n=5):
    """Analyze per-sample results from answers.jsonl."""
    if not os.path.exists(answer_file):
        print(f"Answer file not found: {answer_file}")
        return
    
    records = []
    with open(answer_file) as f:
        for line in f:
            records.append(json.loads(line))
    
    print(f"Total answers: {len(records)}")
    
    # Extract per-sample metrics
    sample_data = []
    for r in records:
        meta = r.get("meta", {})
        comp_tokens = meta.get("completion_tokens", 0)
        verify_ct = meta.get("spec_verify_ct", 0)
        al = comp_tokens / verify_ct if verify_ct > 0 else 0
        
        sample_data.append({
            "question_id": r["question_id"],
            "completion_tokens": comp_tokens,
            "prompt_tokens": meta.get("prompt_tokens", 0),
            "cached_tokens": meta.get("cached_tokens", 0),
            "spec_verify_ct": verify_ct,
            "accept_length": al,
        })
    
    if not sample_data:
        return
    
    # Sort by accept_length
    sorted_by_al = sorted(sample_data, key=lambda x: x["accept_length"], reverse=True)
    
    print(f"\n--- Top {top_n} Highest Accept Length ---")
    for s in sorted_by_al[:top_n]:
        print(f"  Q{s['question_id']:4d}: AL={s['accept_length']:.2f} "
              f"(comp={s['completion_tokens']}, verify={s['spec_verify_ct']})")
    
    print(f"\n--- Top {top_n} Lowest Accept Length ---")
    for s in sorted_by_al[-top_n:]:
        print(f"  Q{s['question_id']:4d}: AL={s['accept_length']:.2f} "
              f"(comp={s['completion_tokens']}, verify={s['spec_verify_ct']})")
    
    # Statistics
    import numpy as np
    als = [s["accept_length"] for s in sample_data if s["accept_length"] > 0]
    if als:
        print(f"\n--- Statistics ---")
        print(f"  Mean AL:   {np.mean(als):.3f}")
        print(f"  Std AL:    {np.std(als):.3f}")
        print(f"  Min AL:    {np.min(als):.3f}")
        print(f"  Max AL:    {np.max(als):.3f}")
        print(f"  Median AL: {np.median(als):.3f}")


analyze_answers(ANSWER_FILE)

## 6. 多模型对比评测

MTP 评测的核心价值在于 **对比不同 MTP 训练策略** 的效果。

In [ ]:
# ============================================================
# 多模型对比实验配置
# ============================================================

def create_comparison_experiment(models, bench_id="aiak_bench", num_questions=200):
    """Create multiple experiment configs for model comparison.
    
    Args:
        models: list of dicts with keys: name, path, eagle_steps, eagle_topk, draft_tokens
    """
    experiments = []
    
    for model in models:
        # Generate deploy script
        script = f"""#!/bin/bash
python3 -m sglang.launch_server \\
    --model-path {model['path']} \\
    --tp-size {model.get('tp_size', 8)} \\
    --dp-size {model.get('dp_size', 1)} \\
    --enable-dp-attention \\
    --trust-remote-code \\
    --mem-fraction-static 0.75 \\
    --max-running-requests 256 \\
    --cuda-graph-max-bs 32 \\
    --chunked-prefill-size 16384 \\
    --speculative-algorithm EAGLE \\
    --speculative-num-steps={model.get('eagle_steps', 3)} \\
    --speculative-eagle-topk={model.get('eagle_topk', 1)} \\
    --speculative-num-draft-tokens={model.get('draft_tokens', 4)} \\
    --host 0.0.0.0 --port 30000
"""
        
        # Generate experiment config
        exp = {
            "experiment": {
                "name": f"eval-{model['name']}",
                "continue_on_error": True,
            },
            "service": {
                "command_file": f"/path/to/start_{model['name']}.sh",
                "ready_port": 30000,
                "ready_url": "http://127.0.0.1:30000/health",
                "startup_timeout_sec": 300,
            },
            "tasks": [
                {
                    "id": f"{bench_id}_{model['name']}",
                    "bench": bench_id,
                    "args": {
                        "port": 30000,
                        "num-questions": num_questions,
                        "thinking": model.get("thinking", False),
                    },
                },
            ],
        }
        
        experiments.append({
            "model": model,
            "script": script,
            "experiment": exp,
        })
    
    return experiments


# 定义对比模型
models_to_compare = [
    {
        "name": "ds_v3_eagle_iter1900",
        "path": "/path/to/ds_v3/checkpoints/iter_1900_hf",
        "eagle_steps": 3,
        "eagle_topk": 1,
        "draft_tokens": 4,
    },
    {
        "name": "ds_v3_eagle_iter3000",
        "path": "/path/to/ds_v3/checkpoints/iter_3000_hf",
        "eagle_steps": 3,
        "eagle_topk": 1,
        "draft_tokens": 4,
    },
    {
        "name": "mimo_0507",
        "path": "/path/to/mimo_0507/model_hf",
        "eagle_steps": 3,
        "eagle_topk": 2,  # wider tree
        "draft_tokens": 6,
    },
]

experiments = create_comparison_experiment(models_to_compare)

print(f"Generated {len(experiments)} experiment configs")
for exp in experiments:
    print(f"  - {exp['model']['name']}: steps={exp['model']['eagle_steps']}, topk={exp['model']['eagle_topk']}")

In [ ]:
# ============================================================
# 对比结果可视化
# ============================================================

def compare_results(result_files: dict):
    """Compare results from multiple evaluations.
    
    Args:
        result_files: {model_name: result_file_path}
    """
    results = {}
    for name, path in result_files.items():
        if os.path.exists(path):
            with open(path) as f:
                results[name] = json.load(f)
    
    if not results:
        print("No result files found. Using demo data.")
        # Demo data for illustration
        results = {
            "Model_A (iter 1900)": {"throughput": 1245.3, "accept_length": 2.85, "latency": 45.2},
            "Model_B (iter 3000)": {"throughput": 1567.8, "accept_length": 3.42, "latency": 36.1},
            "Model_C (mimo)": {"throughput": 1823.5, "accept_length": 3.78, "latency": 31.5},
        }
    
    # Print comparison table
    print("\n" + "=" * 70)
    print(f"{'Model':<25} {'Throughput':>12} {'Accept Len':>12} {'Latency':>10}")
    print("-" * 70)
    for name, r in results.items():
        tp = r.get("throughput", 0)
        al = r.get("accept_length", 0)
        lat = r.get("latency", 0)
        print(f"{name:<25} {tp:>10.1f}/s {al:>10.3f} {lat:>8.1f}s")
    print("=" * 70)
    
    # Relative comparison (first model as baseline)
    if len(results) > 1:
        baseline = list(results.values())[0]
        baseline_name = list(results.keys())[0]
        print(f"\nRelative to baseline ({baseline_name}):")
        for name, r in results.items():
            if name == baseline_name:
                continue
            tp_ratio = r.get("throughput", 0) / baseline.get("throughput", 1)
            al_ratio = r.get("accept_length", 0) / baseline.get("accept_length", 1)
            lat_ratio = r.get("latency", 1) / baseline.get("latency", 1)
            print(f"  {name}: TP {tp_ratio:.2f}x, AL {al_ratio:.2f}x, Latency {lat_ratio:.2f}x")


# Demo comparison
compare_results({})

## 7. 常见问题排查

### 7.1 OOM (Out of Memory)

| 症状 | 解决方案 |
|------|----------|
| CUDA OOM during loading | 减小 `--mem-fraction-static` 到 0.7 或更低 |
| OOM during inference | 减小 `--max-running-requests` |
| OOM with large context | 减小 `--max-total-tokens` |

### 7.2 端口冲突

```bash
# 检查端口占用
lsof -i :30000

# 或
ss -tlnp | grep 30000

# 杀死占用进程
kill -9 <PID>
```

### 7.3 服务启动超时

| 原因 | 解决方案 |
|------|----------|
| 模型太大加载慢 | 增加 `startup_timeout_sec` 到 600+ |
| 多线程加载 | 添加 `--model-loader-extra-config '{"enable_multithread_load": "true", "num_threads": 64}'` |
| NCCL 初始化慢 | 检查 GPU 互联拓扑 |

### 7.4 Accept Length 异常低

| 原因 | 解决方案 |
|------|----------|
| 未启用推测解码 | 检查 `--speculative-algorithm EAGLE` 是否生效 |
| MTP Head 未训练好 | 检查模型 checkpoint 是否包含 MTP weights |
| Temperature 过高 | 使用 `temperature=0` 评测 |
| 输出太短 | 样本 completion_tokens < 10 会导致 AL 不准 |

In [ ]:
# ============================================================
# 诊断工具：快速检查服务状态
# ============================================================

def diagnose_server(host="127.0.0.1", port=30000):
    """Run quick diagnostics on a running SGLang server."""
    base_url = f"http://{host}:{port}"
    
    print("=== Server Diagnostics ===")
    
    # 1. Health check
    try:
        resp = requests.get(f"{base_url}/health", timeout=5)
        print(f"[OK] Health: {resp.status_code}")
    except Exception as e:
        print(f"[FAIL] Health: {e}")
        return
    
    # 2. Model info
    try:
        resp = requests.get(f"{base_url}/v1/models", timeout=5)
        models = resp.json()
        model_id = models.get("data", [{}])[0].get("id", "unknown")
        print(f"[OK] Model: {model_id}")
    except Exception as e:
        print(f"[WARN] Model info: {e}")
    
    # 3. Quick inference test
    try:
        payload = {
            "model": "default",
            "messages": [{"role": "user", "content": "Hi"}],
            "max_tokens": 16,
            "temperature": 0,
        }
        resp = requests.post(f"{base_url}/v1/chat/completions", json=payload, timeout=30)
        result = resp.json()
        
        if "choices" in result:
            usage = result.get("usage", {})
            print(f"[OK] Inference works")
            print(f"     Completion tokens: {usage.get('completion_tokens', '?')}")
            
            # Check for speculative decoding
            if "spec_verify_ct" in str(result):
                print(f"[OK] Speculative decoding detected")
            else:
                print(f"[WARN] No spec_verify_ct in response (speculative decoding may not be active)")
        else:
            print(f"[FAIL] Inference: {result}")
    except Exception as e:
        print(f"[FAIL] Inference: {e}")


# Run diagnostics
# diagnose_server("127.0.0.1", 30000)

## 8. 清理：停止服务

In [ ]:
# qf_mtp_eval 中的清理脚本 (deploy_scripts/ds_align/stop_sglang_cleanup.sh)
# 原理：找到 sglang.launch_server 进程并杀掉

def stop_sglang_server(port=30000):
    """Stop SGLang server by port."""
    import subprocess
    
    # Find process using the port
    try:
        result = subprocess.run(
            ["lsof", "-t", f"-i:{port}"],
            capture_output=True, text=True
        )
        pids = result.stdout.strip().split("\n")
        pids = [p for p in pids if p]
        
        if pids:
            for pid in pids:
                os.kill(int(pid), signal.SIGTERM)
            print(f"Sent SIGTERM to PIDs: {pids}")
            time.sleep(3)
            
            # Check if still running
            for pid in pids:
                try:
                    os.kill(int(pid), 0)  # check if alive
                    os.kill(int(pid), signal.SIGKILL)
                    print(f"Force killed PID {pid}")
                except ProcessLookupError:
                    pass
        else:
            print(f"No process found on port {port}")
    except Exception as e:
        print(f"Error stopping server: {e}")


# Uncomment to stop:
# stop_sglang_server(PORT)

## 9. 完整工作流总结

```
┌─────────────────────────────────────────────────────────────┐
│              MTP 评测完整工作流                               │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  1. 准备阶段                                                │
│     ├── 编写 deploy script (SGLang 启动参数)                │
│     ├── 选择 benchmark (aiak/math/ceval/...)                │
│     └── 编写 experiment.json                                │
│                                                             │
│  2. 执行阶段 (两种方式)                                     │
│     A. 自动: python3 runner.py run-experiment --config ...   │
│     B. 手动: 终端启动服务 → notebook 运行 bench             │
│                                                             │
│  3. 分析阶段                                                │
│     ├── 查看 result.json (throughput, AL, latency)          │
│     ├── 分析 answers.jsonl (per-sample metrics)             │
│     └── 多模型对比 (表格 / 图表)                            │
│                                                             │
│  4. 清理阶段                                                │
│     └── 停止 SGLang 服务                                    │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

## 本节小结

| 知识点 | 掌握内容 |
|--------|----------|
| deploy script | 编写 SGLang 启动脚本, 包含所有推测解码参数 |
| experiment.json | service (部署) + tasks (评测) 的完整配置 |
| runner.py | `run-experiment` (全自动) / `run-bench` (仅评测) |
| 结果解析 | result.json (汇总) + answers.jsonl (逐样本) |
| 多模型对比 | 构建对比实验，分析 AL/Throughput 差异 |
| 问题排查 | OOM / 端口 / 超时 / AL 异常的诊断方法 |

---

## 恭喜！你已完成全部教程

通过这 6 节教程，你已经掌握了：

1. **SGLang 基础** — 启动服务、HTTP/SDK 调用
2. **批量推理** — run_batch、meta_info、性能计算
3. **推测解码** — EAGLE 原理、MTP 关系、Accept Length
4. **项目架构** — 三阶段流程、模块关系、配置机制
5. **Benchmark 开发** — 脚本解析、指标体系、自定义 bench
6. **端到端实验** — experiment.json、runner、结果分析、对比评测

### 参考资源

- SGLang GitHub: https://github.com/sgl-project/sglang
- EAGLE Paper: https://arxiv.org/abs/2401.15077
- qf_mtp_eval 源码: `/mnt/cfs_bj_mt/workspace/limengjie03/tool_chain/qf_mtp_eval`